# A3/A4 Data Cleaning & Entity Resolution (CPU)

Menjalankan *cleaning* dan *entity resolution* secara reproduktif lewat
`sipature_ml` (tanpa duplikasi logika). Ikuti `docs/cleaning-entity-resolution-report.md`
dan `docs/reproducibility-runbook.md` sebelum eksekusi.

Input: raw CSV (dari notebook `01`). Output: `data/interim/*` (hasil cleaning),
`data/processed/*` (canonical destinations + links), dan report + figure.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DATASET_SOURCE_DIR = DRIVE_ROOT / "data" / "raw"  # raw CSV sumber di Drive

PROJECT_DIR = Path("/content/hackathon/ml")
LOCAL_DATASET_DIR = PROJECT_DIR / "data" / "raw"

INTERIM_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"
FIGURE_DIR = PROJECT_DIR / "artifacts" / "figures" / "cleaning-entity"

DRIVE_INTERIM_DIR = DRIVE_ROOT / "data" / "interim"
DRIVE_PROCESSED_DIR = DRIVE_ROOT / "data" / "processed"
DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
DRIVE_FIGURE_DIR = DRIVE_ROOT / "figures" / "cleaning-entity"

SOURCE_ENCODING = "utf-8-sig"

print("Drive root:", DRIVE_ROOT)
print("Sumber dataset:", DATASET_SOURCE_DIR)
print("Interim  :", INTERIM_DIR)
print("Processed:", PROCESSED_DIR)


In [ ]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


In [ ]:
%cd /content/hackathon/ml
!git log --oneline -3


In [ ]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


In [ ]:
import numpy
import pandas
import pyarrow
import rapidfuzz
import matplotlib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("RapidFuzz:", rapidfuzz.__version__)
print("Matplotlib:", matplotlib.__version__)


In [ ]:
# Salin raw CSV sumber dari Drive ke lokal (path deterministik untuk sipature_ml).
import shutil
from pathlib import Path

LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_SOURCE_DIR.is_dir(), (
    f"Sumber dataset tidak ditemukan di Drive: {DATASET_SOURCE_DIR}\n"
    "Unggah CSV mentah ke folder tersebut sebelum melanjutkan."
)

copied = []
for source in sorted(DATASET_SOURCE_DIR.glob("*.csv")):
    destination = LOCAL_DATASET_DIR / source.name
    shutil.copy2(source, destination)
    copied.append(source.name)
    print("Disalin:", source.name)

print("\nTotal file:", len(copied))


In [ ]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


In [ ]:
from sipature_ml.config import load_config, ML_ROOT

config = load_config("pipeline")

# File adjudication manusia untuk entity resolution (harus ada di repo).
entity_review = ML_ROOT / "configs" / "entity-review-v1.csv"
assert entity_review.is_file(), f"entity-review-v1.csv tidak ditemukan: {entity_review}"

print("Pipeline version:", config["pipeline_version"])
print("Entity review file:", entity_review)
print("Auto-match name similarity:", config["entity_resolution"]["auto_match_name_similarity"])
print("Manual-review name similarity:", config["entity_resolution"]["manual_review_name_similarity"])
print("Auto-match max distance (m):", config["entity_resolution"]["auto_match_max_distance_meters"])


In [ ]:
from sipature_ml.cleaning import run_cleaning

cleaning_summary = run_cleaning(LOCAL_DATASET_DIR, INTERIM_DIR, REPORT_DIR)

print("Raw records          :", cleaning_summary["reviews"]["raw_records"])
print("Duplicate excess rmv :", cleaning_summary["reviews"]["exact_duplicate_excess_removed"])
print("Empty records excl   :", cleaning_summary["reviews"]["empty_records_excluded"])
print("Clean records        :", cleaning_summary["reviews"]["clean_records"])
print("Clean textual records:", cleaning_summary["reviews"]["clean_textual_records"])
print("Place source records :", cleaning_summary["place_source_records"])
print("Interim outputs:")
for name in sorted(cleaning_summary["outputs"]):
    print("  -", name)


In [ ]:
from sipature_ml.entity_resolution import run_entity_resolution

resolution_summary = run_entity_resolution(INTERIM_DIR, PROCESSED_DIR, REPORT_DIR)

print("Canonical destinations       :", resolution_summary["canonical_destinations"])
print("Metadata anchor destinations :", resolution_summary["metadata_anchor_destinations"])
print("Unresolved placeholder       :", resolution_summary["unresolved_placeholder_destinations"])
print("Source links                 :", resolution_summary["source_links"])
print("Link status counts           :", resolution_summary["link_status_counts"])
print("Ambiguous candidate rows     :", resolution_summary["ambiguous_candidate_rows"])
print("Unresolved source rows       :", resolution_summary["unresolved_source_rows"])
print("All reviews have destination_id:", resolution_summary["all_reviews_have_destination_id"])
print("Processed outputs:")
for name in sorted(resolution_summary["outputs"]):
    print("  -", name)


In [ ]:
from sipature_ml.quality_figures import generate_quality_figures

figures = generate_quality_figures(REPORT_DIR, PROCESSED_DIR, FIGURE_DIR)

print("Figures:", len(figures))
for name in figures:
    print("-", name)


In [ ]:
# Salin output interim + processed + report + figure ke Drive (artefak persisten).
import shutil
from pathlib import Path

for local_dir, drive_dir in (
    (INTERIM_DIR, DRIVE_INTERIM_DIR),
    (PROCESSED_DIR, DRIVE_PROCESSED_DIR),
    (REPORT_DIR, DRIVE_REPORT_DIR),
    (FIGURE_DIR, DRIVE_FIGURE_DIR),
):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for source in sorted(local_dir.glob("*")):
        if source.is_file():
            shutil.copy2(source, drive_dir / source.name)
            print(f"Disalin: {source.name} -> {drive_dir}")


In [ ]:
# ============================================================
# RUN SUMMARY — output path, hash sumber, dan metrics.
# ============================================================
import json
from pathlib import Path

print("CLEANING VERSION :", cleaning_summary["cleaning_version"])
print("ENTITY RES VERSION:", resolution_summary["entity_resolution_version"])

print("\nSOURCE HASHES:")
for name, digest in sorted(cleaning_summary["source_files"].items()):
    print(f"  {digest}  {name}")

print("\nOUTPUT INTERIM DIR  :", INTERIM_DIR)
print("OUTPUT PROCESSED DIR :", PROCESSED_DIR)
print("OUTPUT REPORT DIR    :", REPORT_DIR)
print("OUTPUT FIGURE DIR    :", FIGURE_DIR)
print("DRIVE INTERIM DIR    :", DRIVE_INTERIM_DIR)
print("DRIVE PROCESSED DIR  :", DRIVE_PROCESSED_DIR)

metrics_path = REPORT_DIR / "entity_resolution_metrics.json"
if metrics_path.is_file():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    pre = metrics["pre_adjudication"]
    print("\nENTITY REVIEW METRICS (pre-adjudication):")
    print("  reviewed pairs:", metrics["reviewed_pairs"])
    print("  precision:", pre["precision"], " recall:", pre["recall"], " f1:", pre["f1"])
    print("  false_merge_rate_among_predicted_matches:", pre["false_merge_rate_among_predicted_matches"])
